In [4]:
import pandas as pd
import matplotlib.pyplot as plt 

In [ ]:
# Loading the CSV file into a DataFrame called df
df = pd.read_csv('data/data.csv')

# Showing first 5 rows
print(df.head())

# Showing all column names
print(df.columns)

# Showing total rows and columns
print(df.shape)

In [ ]:
# Showing data types and memory info
df.info()

In [ ]:
# Basic statistics of all numeric columns
df.describe()

In [ ]:
# Counting missing values in each column
print(df.isnull().sum())

In [ ]:
# Keeping only the columns we need for analysis
df = df[['country', 'year', 'co2', 'co2_per_capita', 'population', 'gdp']]

# Filtering only from 1990 onwards — older data has too many NaN values
df = df[df['year'] >= 1990]

# Removing rows where CO2 value is missing
df = df.dropna(subset=['co2'])

# Confirming new shape after cleaning
print("Cleaned data shape:", df.shape)

# Confirming no missing CO2 values remain
print("Missing CO2 values:", df['co2'].isnull().sum())

# Preview cleaned data
print(df.head())

In [ ]:
# Grouping by country and summing all CO2 emissions since 1990
total_by_country = df.groupby('country')['co2'].sum()

# Sorting highest to lowest and taking top 10
top10 = total_by_country.sort_values(ascending=False).head(10)

# Printing result
print(top10)

In [ ]:
# List of non-country entries to exclude
exclude = [
    'World', 'Asia', 'Europe', 'Africa', 'Oceania',
    'North America', 'South America', 'Antarctica',
    'Asia (GCP)', 'North America (GCP)', 'South America (GCP)',
    'Europe (GCP)', 'Africa (GCP)', 'Oceania (GCP)',
    'High-income countries', 'Low-income countries',
    'Upper-middle-income countries', 'Lower-middle-income countries',
    'OECD (GCP)', 'Non-OECD (GCP)', 'International transport'
]

# Filtering out non-country rows
df_countries = df[~df['country'].isin(exclude)]

# Now getting top 10 actual countries
total_by_country = df_countries.groupby('country')['co2'].sum()
top10 = total_by_country.sort_values(ascending=False).head(10)

print(top10)

In [ ]:
# Stronger filter — remove anything with brackets, 
# 'excl', 'EU', 'GCP', 'income', 'OECD'
df_countries = df[~df['country'].str.contains(
    'excl|GCP|income|OECD|European Union|Middle East|World|'
    'Asia|Europe|Africa|Oceania|America|Antarctica|transport',
    case=False
)]

# Now getting top 10 actual countries
total_by_country = df_countries.groupby('country')['co2'].sum()
top10 = total_by_country.sort_values(ascending=False).head(10)

print(top10)

In [19]:
import os

# Creating visuals folder if it doesn't exist
os.makedirs('visuals', exist_ok=True)

In [ ]:

# Creating bar chart for top 10 countries
top10.plot(kind='bar', color='crimson', figsize=(12, 6))

# Adding title
plt.title('Top 10 CO2 Emitting Countries (1990-2023)', fontsize=14)

# Adding axis labels
plt.xlabel('Country', fontsize=12)
plt.ylabel('Total CO2 Emissions (Million Tonnes)', fontsize=12)

# Rotating x labels so names don't overlap
plt.xticks(rotation=45)

# Adjusting layout
plt.tight_layout()

# Saving chart
plt.savefig('visuals/top10_countries.png')

# Displaying chart
plt.show()

In [ ]:
# Grouping by year and summing CO2 across all real countries
global_trend = df_countries.groupby('year')['co2'].sum()

# Line chart — emissions trend over time
global_trend.plot(kind='line', color='steelblue', figsize=(12, 6))

plt.title('Global CO2 Emissions Over Time (1990-2023)', fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Total CO2 Emissions (Million Tonnes)', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.savefig('visuals/global_trend.png')
plt.show()

In [ ]:
# Filtering data for Nepal only
nepal = df_countries[df_countries['country'] == 'Nepal']

# Plotting Nepal's emissions over time
plt.figure(figsize=(12, 6))
plt.plot(nepal['year'], nepal['co2'], color='green', linewidth=2, label='Nepal')

plt.title("Nepal's CO2 Emissions Over Time (1990-2023)", fontsize=14)
plt.xlabel('Year', fontsize=12)
plt.ylabel('CO2 Emissions (Million Tonnes)', fontsize=12)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('visuals/nepal_trend.png')
plt.show()

In [ ]:
# Finding which year had highest global CO2 emissions
peak_year = global_trend.idxmax()
peak_value = global_trend.max()

print(f"Highest emission year: {peak_year}")
print(f"Total emissions that year: {peak_value:.2f} Million Tonnes")

In [ ]:
# Better visualization — line chart with peak highlighted
fig, ax = plt.subplots(figsize=(14, 6))

# Plot line
ax.plot(global_trend.index, global_trend.values, 
        color='steelblue', linewidth=2.5, zorder=2)

# Fill area under line
ax.fill_between(global_trend.index, global_trend.values, 
                alpha=0.3, color='steelblue')

# Highlight peak year with red dot
ax.scatter(peak_year, peak_value, 
           color='crimson', s=120, zorder=5, label=f'Peak: {peak_year}')

# Add annotation label on peak
ax.annotate(f'Peak: {peak_year}\n{peak_value:.0f} MT',
            xy=(peak_year, peak_value),
            xytext=(peak_year - 6, peak_value - 2000),
            fontsize=10, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

ax.set_title('Global CO2 Emissions (1990-2024) — Record High in 2024', fontsize=14)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total CO2 Emissions (Million Tonnes)', fontsize=12)
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig('visuals/peak_year.png')
plt.show()

In [ ]:
# Simplified CO2 vs GDP — easier to understand
fig, ax = plt.subplots(figsize=(14, 7))

# Plot scatter with better sizing
scatter = ax.scatter(df_gdp['gdp'] / 1e12,  # Convert to Trillions
                     df_gdp['co2'], 
                     alpha=0.4, 
                     color='purple', 
                     s=15)

# Add clear trend line
from numpy.polynomial.polynomial import polyfit
import numpy as np

x = df_gdp['gdp'] / 1e12
y = df_gdp['co2']
b, m = polyfit(x, y, 1)
ax.plot(sorted(x), [b + m * xi for xi in sorted(x)], 
        color='red', linewidth=2, label='Trend Line')

# Add zone labels
ax.text(0.5, 9000, '🌍 Poor countries\nlow emissions', 
        fontsize=10, color='green')
ax.text(15, 11000, '🏭 Rich countries\nhigh emissions', 
        fontsize=10, color='red')

ax.set_title('Do Richer Countries Emit More CO2?', fontsize=16, fontweight='bold')
ax.set_xlabel('GDP (Trillion USD) →  Richer', fontsize=12)
ax.set_ylabel('CO2 Emissions (Million Tonnes) →  More Pollution', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visuals/co2_vs_gdp.png')
plt.show()

In [ ]:
# Saving all charts in square Instagram format

# Chart 1 — Top 10 Countries
fig, ax = plt.subplots(figsize=(8, 8))
top10.plot(kind='bar', color='crimson', ax=ax)
ax.set_title('Top 10 CO2 Emitting Countries (1990-2024)', fontsize=13)
ax.set_xlabel('Country')
ax.set_ylabel('Total CO2 Emissions (Million Tonnes)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/ig_top10.png', dpi=150)
plt.show()

In [ ]:
# Chart 2 — Global Trend
fig, ax = plt.subplots(figsize=(8, 8))
global_trend.plot(kind='line', color='steelblue', linewidth=2.5, ax=ax)
ax.set_title('Global CO2 Emissions Over Time (1990-2024)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Total CO2 Emissions (Million Tonnes)')
ax.grid(True)
plt.tight_layout()
plt.savefig('visuals/ig_global_trend.png', dpi=150)
plt.show()

# Chart 3 — Nepal
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(nepal['year'], nepal['co2'], color='green', linewidth=2.5)
ax.set_title("Nepal's CO2 Emissions Over Time (1990-2024)", fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('CO2 Emissions (Million Tonnes)')
ax.grid(True)
plt.tight_layout()
plt.savefig('visuals/ig_nepal.png', dpi=150)
plt.show()

# Chart 4 — Peak Year Square Instagram Version
fig, ax = plt.subplots(figsize=(8, 8))

# Plot line
ax.plot(global_trend.index, global_trend.values,
        color='steelblue', linewidth=2.5, zorder=2)

# Fill area under line
ax.fill_between(global_trend.index, global_trend.values,
                alpha=0.3, color='steelblue')

# Red dot on peak
ax.scatter(peak_year, peak_value,
           color='crimson', s=120, zorder=5, label=f'Peak: {peak_year}')

# Annotation
ax.annotate(f'Peak: {peak_year}\n{peak_value:.0f} MT',
            xy=(peak_year, peak_value),
            xytext=(peak_year - 8, peak_value - 3000),
            fontsize=10, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

ax.set_title('Global CO2 Emissions (1990-2024) — Record High in 2024', fontsize=13)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Total CO2 Emissions (Million Tonnes)', fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('visuals/ig_peak.png', dpi=150)
plt.show()

# Chart 5 — CO2 vs GDP with trend line — Square Instagram Version
import numpy as np

fig, ax = plt.subplots(figsize=(8, 8))

# Scatter plot
ax.scatter(df_gdp['gdp'] / 1e12, df_gdp['co2'],
           alpha=0.4, color='purple', s=15)

# Trend line
x = df_gdp['gdp'] / 1e12
y = df_gdp['co2']
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, p(x_line),
        color='red', linewidth=2, label='Trend Line')

# Zone labels
ax.text(0.5, 9000, '🌍 Poor countries\nlow emissions',
        fontsize=10, color='green')
ax.text(15, 11000, '🏭 Rich countries\nhigh emissions',
        fontsize=10, color='red')

ax.set_title('Do Richer Countries Emit More CO2?', fontsize=13, fontweight='bold')
ax.set_xlabel('GDP (Trillion USD) → Richer', fontsize=11)
ax.set_ylabel('CO2 Emissions (Million Tonnes) → More Pollution', fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visuals/ig_gdp.png', dpi=150)
plt.show()